# Classificação de Gripe com K-Nearest Neighbors (KNN)

**Objetivo**: Prever se uma pessoa ficou gripada no ano passado com base em seus hábitos de saúde e estilo de vida, utilizando o algoritmo KNN.

---

### O que é o KNN?

O **K-Nearest Neighbors** é um algoritmo de classificação simples. Para classificar um novo ponto, ele calcula a **distância** até todos os outros exemplos do dataset, seleciona os **K mais próximos** e retorna a classe que aparece **com mais frequência** entre eles.

## Importação das bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

---
## 1. Carregamento dos dados

In [ ]:
df = pd.read_csv("https://docs.google.com/spreadsheets/d/1g1aQ61vijh6uHJuc8sijeBEMsoIQ2a5yLwUK04Wptlg/export?format=csv")

df.columns = [
    'timestamp',
    'gripado',
    'vacina',
    'ambientes_lotados',
    'viagem',
    'alergia',
    'horas_sono',
    'atividade_fisica',
    'alimentacao',
    'lavagem_maos',
    'nivel_estresse'
]

df = df.drop(columns=['timestamp'])

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip().str.lower()

df['vacina']            = df['vacina'].map({'sim': 1, 'não': 0})
df['ambientes_lotados'] = df['ambientes_lotados'].map({'sim': 1, 'não': 0})
df['atividade_fisica']  = df['atividade_fisica'].map({'sim': 1, 'não': 0})
df['viagem']            = df['viagem'].map({'nuca': 0, 'poucas vezes por ano': 1, 'pelo menos uma vez por mês': 2})
df['alergia']           = df['alergia'].map({'não': 0, 'pouco': 1, 'médio': 2, 'muito': 3})
df['horas_sono']        = df['horas_sono'].map({'4 horas ou menos': 0, 'entre 4 e 6 horas': 1, 'mais de 6 horas': 2})
df['alimentacao']       = df['alimentacao'].map({'não, raramente': 0, 'às vezes': 1, 'sim, a maior parte do tempo': 2})
df['lavagem_maos']      = df['lavagem_maos'].map({'2 vezes ou menos': 0, '3 a 5 vezes': 1, '6 a 10 vezes': 2, 'mais de 10 vezes': 3})
df['nivel_estresse']    = df['nivel_estresse'].fillna(df['nivel_estresse'].median())

df.head()

### 1.1 Distribuição do Target

In [ ]:
count = df['gripado'].value_counts()
labels = ['Gripado', 'Não Gripado']

fig, axes = plt.subplots(figsize=(8, 5))
bars = axes.bar(labels, count.values, linewidth=1.5, width=0.5)

for bar in bars:
    axes.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.5,
        str(int(bar.get_height())),
        ha='center', fontsize=13, fontweight='bold'
    )

axes.set_ylabel('Quantidade de Pessoas')
axes.set_ylim(0, count.max() + 18)
axes.spines[['top', 'right']].set_visible(False)

axes.set_title('Distribuição do Target', fontsize=15, fontweight='bold', pad=15, color='#222222')
plt.show()

---
## 2. Escolha e Preparação das Features

In [ ]:
y_raw = df['gripado']
X = df.drop(columns=['gripado'])

label_encoder = LabelEncoder()

y = np.asarray(label_encoder.fit_transform(y_raw))

print('Classes:', list(label_encoder.classes_))
print()
print(f'Features utilizadas:')

for i, col in enumerate(X.columns, 1):
    print(f'{i}. {col}')

---
## 3. Separação em Treino e Teste

- **Treino (80%)**
- **Teste (20%)**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f'Treino : {X_train.shape[0]} amostras ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Teste  : {X_test.shape[0]} amostras ({X_test.shape[0]/len(X)*100:.0f}%)')
print()

print(f'Proporção de gripados no treino: {y_train.mean():.2f}')
print(f'Proporção de gripados no teste : {y_test.mean():.2f}')

---
## 4. Treinamento com o KNN

In [ ]:
model = KNeighborsClassifier(n_neighbors=2, weights='distance')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

---
## 5. Avaliação do Modelo

In [ ]:
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_)

print(f'Acurácia: {acc:.4f}')
print('\nMatriz de confusão:')
print(cm)
print('\nRelatório de classificação:')
print(report)

In [ ]:
print('\nExemplos de previsões:')
sample = pd.DataFrame({
    'Real'     : label_encoder.inverse_transform(y_test),
    'Previsto' : label_encoder.inverse_transform(y_pred),
}).reset_index(drop=True)

print(sample.head(10).to_string(index=False))